In [21]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

# Import Model
import xgboost as xgb

# Import Metrics
from sklearn.metrics import mean_absolute_error as mae
from sklearn.metrics import root_mean_squared_error as rmse

In [22]:
df = pd.read_csv("/home/katherine-schwind/Documents/Erdos/summer26-patient-length-of-stay/new_cleaned_data/PUDF_base1_2q2017_cleaned/PUDF_base1_4q2019_cleaned.csv", low_memory=False)
df=df.drop(df[df['SOURCE_OF_ADMISSION'] == '`'].index)

df.info()

<class 'pandas.DataFrame'>
Index: 587322 entries, 0 to 587526
Data columns (total 45 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   RECORD_ID             587322 non-null  int64  
 1   THCIC_ID              587322 non-null  int64  
 2   TYPE_OF_ADMISSION     587322 non-null  float64
 3   SOURCE_OF_ADMISSION   587322 non-null  str    
 4   PUBLIC_HEALTH_REGION  587322 non-null  float64
 5   PAT_STATUS            587322 non-null  int64  
 6   SEX_CODE              587322 non-null  str    
 7   RACE                  587322 non-null  int64  
 8   ETHNICITY             587322 non-null  int64  
 9   ADMIT_WEEKDAY         587322 non-null  float64
 10  LENGTH_OF_STAY        587322 non-null  float64
 11  PAT_AGE               587322 non-null  int64  
 12  EMERGENCY_DEPT_FLAG   587322 non-null  str    
 13  DIAG_CODES_OA         587322 non-null  str    
 14  CODE_1                587322 non-null  int64  
 15  CODE_2          

In [25]:
df['PROLONGED'] = [1 if los >= 30 else 0 for los in df["LENGTH_OF_STAY"]]
df['NUM_CODES'] = df.iloc[:, 18:40].sum(axis = 1)
df['strata'] = df['QUARTER'].astype(str) + '_' + df['PAT_RURAL'].astype(str) + '_' + df['PROVIDER_RURAL'].astype(str)

df.info()

<class 'pandas.DataFrame'>
Index: 587322 entries, 0 to 587526
Data columns (total 48 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   RECORD_ID             587322 non-null  int64  
 1   THCIC_ID              587322 non-null  int64  
 2   TYPE_OF_ADMISSION     587322 non-null  float64
 3   SOURCE_OF_ADMISSION   587322 non-null  str    
 4   PUBLIC_HEALTH_REGION  587322 non-null  float64
 5   PAT_STATUS            587322 non-null  int64  
 6   SEX_CODE              587322 non-null  str    
 7   RACE                  587322 non-null  int64  
 8   ETHNICITY             587322 non-null  int64  
 9   ADMIT_WEEKDAY         587322 non-null  float64
 10  LENGTH_OF_STAY        587322 non-null  int64  
 11  PAT_AGE               587322 non-null  int64  
 12  EMERGENCY_DEPT_FLAG   587322 non-null  str    
 13  DIAG_CODES_OA         587322 non-null  str    
 14  CODE_1                587322 non-null  int64  
 15  CODE_2          

In [24]:
df.LENGTH_OF_STAY = df.LENGTH_OF_STAY.astype(int)

In [8]:
cat_features = (
    df.columns[2:3].tolist() +
    df.columns[6:13].tolist() +
    df.columns[14:40].tolist() +
    df.columns[41:45].tolist() +
    df.columns[46:47].tolist()
)
cat_features

['TYPE_OF_ADMISSION',
 'SEX_CODE',
 'RACE',
 'ETHNICITY',
 'ADMIT_WEEKDAY',
 'LENGTH_OF_STAY',
 'PAT_AGE',
 'EMERGENCY_DEPT_FLAG',
 'CODE_1',
 'CODE_2',
 'CODE_3',
 'CODE_4',
 'CODE_5',
 'CODE_6',
 'CODE_7',
 'CODE_8',
 'CODE_9',
 'CODE_10',
 'CODE_11',
 'CODE_12',
 'CODE_13',
 'CODE_14',
 'CODE_15',
 'CODE_16',
 'CODE_17',
 'CODE_18',
 'CODE_19',
 'CODE_20',
 'CODE_21',
 'CODE_22',
 'PAT_RURAL',
 'PROVIDER_RURAL',
 'PAT2PROV_DISTANCE',
 'QUARTER',
 'PAT_LATITUDE',
 'PAT_LONGITUDE',
 'PROVIDER_LATITUDE',
 'PROVIDER_LONGITUDE',
 'NUM_CODES']

In [11]:
df[cat_features] = (
    df[cat_features]
    .fillna("Missing")
    .astype('category')
)

In [13]:
X_train, X_test, y_train, y_test = train_test_split(df[cat_features],df.LENGTH_OF_STAY, shuffle = True, random_state = 2222, stratify = df['strata'], test_size = .2)

In [14]:
strata_train = df.loc[X_train.index, 'strata']

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=2222
)

In [15]:
from sklearn.preprocessing import LabelEncoder


In [ ]:
mae_scores = []

# Fit the model for each fold
for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train, strata_train)):
    X_tr = X_train.iloc[train_idx].copy()
    X_val = X_train.iloc[valid_idx].copy()
    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[valid_idx]

    le = LabelEncoder()
    y_tr_encoded = le.fit_transform(y_tr)
    y_val_encoded = le.fit_transform(y_val)

    # Instantiate Model
    model = xgb.XGBRegressor(
        n_estimators=1000,
        max_depth=6, eval_metric='mae',
        learning_rate=0.03, early_stopping_rounds=400, 
        random_state=42, enable_categorical=True         
    )

    model.fit(X_tr, y_tr_encoded, eval_set=[(X_val, y_val_encoded)], verbose=50)

    # Find predictions and store rmses  
    preds = model.predict(X_val)
    
    mae_pred = mae(y_val_encoded,preds)

    mae_scores.append(mae_pred)

    print(f"Fold {fold+1}: MAE = {mae_pred:.3f}")

[0]	validation_0-mae:3.79930
[50]	validation_0-mae:0.84702
[100]	validation_0-mae:0.20629
[150]	validation_0-mae:0.06976
[200]	validation_0-mae:0.04104
[250]	validation_0-mae:0.03481
[300]	validation_0-mae:0.03354
[350]	validation_0-mae:0.03321
[400]	validation_0-mae:0.03312
[450]	validation_0-mae:0.03311
[500]	validation_0-mae:0.03310
[550]	validation_0-mae:0.03310
[600]	validation_0-mae:0.03309
[650]	validation_0-mae:0.03309
[700]	validation_0-mae:0.03309
[750]	validation_0-mae:0.03309
[800]	validation_0-mae:0.03309
[850]	validation_0-mae:0.03309
[900]	validation_0-mae:0.03309
[950]	validation_0-mae:0.03309
[999]	validation_0-mae:0.03309
Fold 1: MAE = 0.033
[0]	validation_0-mae:3.82420
[50]	validation_0-mae:0.87576
[100]	validation_0-mae:0.23203
[150]	validation_0-mae:0.09399
[200]	validation_0-mae:0.06481
[250]	validation_0-mae:0.05849
[300]	validation_0-mae:0.05713
[350]	validation_0-mae:0.05685
[400]	validation_0-mae:0.05679
[450]	validation_0-mae:0.05677
[500]	validation_0-mae:0.

In [18]:
rmse_scores = []

# Fit the model for each fold
for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train, strata_train)):
    X_tr = X_train.iloc[train_idx].copy()
    X_val = X_train.iloc[valid_idx].copy()
    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[valid_idx]

    le = LabelEncoder()
    y_tr_encoded = le.fit_transform(y_tr)
    y_val_encoded = le.fit_transform(y_val)

    # Instantiate Model
    model = xgb.XGBRegressor(
        n_estimators=1000,
        max_depth=6, eval_metric='rmse',
        learning_rate=0.03, early_stopping_rounds=400, 
        random_state=42, enable_categorical=True         
    )

    model.fit(X_tr, y_tr_encoded, eval_set=[(X_val, y_val_encoded)], verbose=50)

    # Find predictions and store rmses  
    preds = model.predict(X_val)
    
    rmse_pred = rmse(y_val_encoded,preds)

    rmse_scores.append(rmse_pred)

    print(f"Fold {fold+1}: RMSE = {rmse_pred:.3f}")

[0]	validation_0-rmse:7.59877
[50]	validation_0-rmse:2.56675
[100]	validation_0-rmse:2.01756
[150]	validation_0-rmse:1.99027
[200]	validation_0-rmse:1.98979
[250]	validation_0-rmse:1.98991
[300]	validation_0-rmse:1.99012
[350]	validation_0-rmse:1.99017
[400]	validation_0-rmse:1.99014
[450]	validation_0-rmse:1.99013
[500]	validation_0-rmse:1.99012
[550]	validation_0-rmse:1.99012
[574]	validation_0-rmse:1.99012
Fold 1: RMSE = 1.990
[0]	validation_0-rmse:7.87102
[50]	validation_0-rmse:3.22169
[100]	validation_0-rmse:2.71351
[150]	validation_0-rmse:2.67859
[200]	validation_0-rmse:2.67606
[250]	validation_0-rmse:2.67553
[300]	validation_0-rmse:2.67535
[350]	validation_0-rmse:2.67529
[400]	validation_0-rmse:2.67530
[450]	validation_0-rmse:2.67530
[500]	validation_0-rmse:2.67530
[550]	validation_0-rmse:2.67530
[600]	validation_0-rmse:2.67530
[650]	validation_0-rmse:2.67530
[700]	validation_0-rmse:2.67530
[735]	validation_0-rmse:2.67530
Fold 2: RMSE = 2.675
[0]	validation_0-rmse:7.86012
[50]	v